# Homework 10

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [2]:
# function for reporting the results of the grid search and the test set evaluation
def report(name, gs, X_test, y_test):
    y_pred = gs.predict(X_test)
    print(f"{name} – best params: {gs.best_params_}")
    print(f"CV RMSE:   {-gs.best_score_:.3f}")
    print(f"Test R²:   {r2_score(y_test, y_pred):.4f}")
    print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
    print(f"Test MAE:  {mean_absolute_error(y_test, y_pred):.4f}")

In [3]:
# load and sort chronologically (time series!)
poly = pd.read_excel("../data_polymerization.xlsx").sort_values("Date.Time")
X = poly.drop(columns=["Date.Time", "Quality"])   # Date.Time is an index, not a feature
y = poly["Quality"]

# chronological hold-out: last 10 % as test set
n_test = int(0.1 * len(poly))
X_train, X_test = X.iloc[:-n_test], X.iloc[-n_test:]
y_train, y_test = y.iloc[:-n_test], y.iloc[-n_test:]

# pipeline: scale features -> MLP, TransformedTargetRegressor additionally scales y
mlp = Pipeline([
    ("scaler", StandardScaler()),                                 # NNs require scaled inputs
    ("mlp", MLPRegressor(max_iter=3000, early_stopping=True,      # early stopping against overfitting
                         random_state=42)),
])
model = TransformedTargetRegressor(mlp, transformer=StandardScaler())  # scale target too

# tune architecture (incl. deeper variants), regularization, learning rate and activation
param_grid = {
    "regressor__mlp__hidden_layer_sizes": [
        (32,), (64,), (32, 32), (64, 32),          # original grid
        (128, 64), (64, 64, 32), (128, 64, 32),    # deeper/wider variants
    ],
    "regressor__mlp__alpha": [1e-4, 1e-3, 1e-2],
    "regressor__mlp__learning_rate_init": [1e-3, 1e-2],
    "regressor__mlp__activation": ["relu", "tanh"],
}

gs_poly = GridSearchCV(model, param_grid, cv=TimeSeriesSplit(n_splits=5),
                       scoring="neg_root_mean_squared_error", n_jobs=-1)
gs_poly.fit(X_train, y_train)
report("Polymerization", gs_poly, X_test, y_test)

# CV comparison per architecture (for the discussion)
res = pd.DataFrame(gs_poly.cv_results_)
for h in param_grid["regressor__mlp__hidden_layer_sizes"]:
    scores = res[res["param_regressor__mlp__hidden_layer_sizes"].apply(lambda x: x == h)]["mean_test_score"]
    print(f"{h}: best CV RMSE {-scores.max():.3f}")

Polymerization – best params: {'regressor__mlp__activation': 'relu', 'regressor__mlp__alpha': 0.0001, 'regressor__mlp__hidden_layer_sizes': (64, 32), 'regressor__mlp__learning_rate_init': 0.001}
CV RMSE:   0.738
Test R²:   0.9551
Test RMSE: 0.7610
Test MAE:  0.4067
(32,): best CV RMSE 1.355
(64,): best CV RMSE 1.367
(32, 32): best CV RMSE 1.256
(64, 32): best CV RMSE 0.738
(128, 64): best CV RMSE 1.101
(64, 64, 32): best CV RMSE 0.885
(128, 64, 32): best CV RMSE 0.994


In [4]:
man = pd.read_excel("../data_manufacturing.xlsx")
X = man.drop(columns=["Dissolution"])
y = man["Dissolution"]

# separate categorical and numerical features
cat_cols = X.select_dtypes(include=["object", "str"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# scale numerics, one-hot encode categoricals (drop first level to avoid redundancy)
prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first"), cat_cols),
])

# no temporal structure -> random split is fine here
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

mlp = Pipeline([
    ("prep", prep),
    ("mlp", MLPRegressor(solver="lbfgs", max_iter=10000,   # lbfgs: better for small datasets
                         random_state=42)),
])
model = TransformedTargetRegressor(mlp, transformer=StandardScaler())

# tune with small architectures, wide alpha range and two activation functions
param_grid = {
    "regressor__mlp__hidden_layer_sizes": [(8,), (16,), (32,), (16, 8)],
    "regressor__mlp__alpha": [1e-3, 1e-2, 1e-1, 1, 10],
    "regressor__mlp__activation": ["relu", "tanh"],
}

gs_man = GridSearchCV(model, param_grid, cv=KFold(5, shuffle=True, random_state=42),
                      scoring="neg_root_mean_squared_error", n_jobs=-1)
gs_man.fit(X_train, y_train)
report("Manufacturing", gs_man, X_test, y_test)

res = pd.DataFrame(gs_man.cv_results_)
for act in ["relu", "tanh"]:
    scores = res[res["param_regressor__mlp__activation"] == act]["mean_test_score"]
    print(f"{act}: best CV RMSE {-scores.max():.3f}")

Manufacturing – best params: {'regressor__mlp__activation': 'tanh', 'regressor__mlp__alpha': 0.1, 'regressor__mlp__hidden_layer_sizes': (32,)}
CV RMSE:   0.783
Test R²:   0.9546
Test RMSE: 0.7029
Test MAE:  0.4844
relu: best CV RMSE 0.833
tanh: best CV RMSE 0.783


### Discussion: Polymerization

Best configuration: (64, 32) hidden units, ReLU, alpha = 1e-4, learning rate = 0.001. This gives a test RMSE of about 0.76, MAE of about 0.41 and R² of about 0.96.

Comparison with Exercise 10:

| Model | Test RMSE | Test MAE | Test R² |
|---|---|---|---|
| Decision Tree | ≈ 2.47 | ≈ 1.29 | ≈ 0.52 |
| Random Forest | ≈ 1.65 | ≈ 0.70 | ≈ 0.79 |
| MLP | ≈ 0.76 | ≈ 0.41 | ≈ 0.96 |

Here the neural network clearly outperforms the Random Forest, roughly halving both RMSE and MAE. The reason is that with 2708 samples there is enough data for the network to learn a genuinely nonlinear response surface. Part of the RMSE gap also reflects the model classes themselves. The forest's piecewise-constant predictions occasionally jump badly on the chronological test block, whereas the network interpolates smoothly.

Architecture. The grid included deeper and wider variants, and the CV comparison shows a clear plateau at (64, 32):

| Architecture | Best CV RMSE |
|---|---|
| (32,) | 1.355 |
| (64,) | 1.367 |
| (32, 32) | 1.256 |
| (64, 32) | 0.738 |
| (128, 64) | 1.101 |
| (64, 64, 32) | 0.885 |
| (128, 64, 32) | 0.994 |

A second hidden layer helps substantially, but adding further width or depth degrades CV performance again. The model is capacity-sufficient at (64, 32), and larger networks only make the optimization harder and increase overfitting risk on the small early folds of `TimeSeriesSplit`.

The selected `alpha = 1e-4` is the weakest value in the grid and three orders of magnitude weaker than the `alpha = 0.1` selected on Manufacturing, because more data is itself a regularizer. We also enabled `early_stopping=True` here. On Manufacturing this was not applicable, since the `lbfgs` solver performs full-batch optimization and does not use an internal validation split.

### Discussion: Manufacturing

Best configuration: (32,) hidden units, tanh, alpha = 0.1 with the lbfgs solver. This gives a test RMSE of about 0.70, MAE of about 0.48 and R² of about 0.95.

Despite only 81 training samples, the MLP reaches a strong test fit, but only under conditions the small-data boundary allows: a single small hidden layer, strong L2 regularisation, and the full-batch `lbfgs` solver instead of `adam`. Cross-validation slightly preferred tanh over ReLU (CV RMSE 0.783 vs 0.833). With this little data the smooth, bounded tanh is a plausible winner, but the margin is small and should not be over-interpreted.

### Overall conclusions

1. Data volume decides how much a neural network gains. On the 2708-sample Polymerization set the MLP halves the Random Forest's test error. On the 90-sample Manufacturing set it works, but only as a heavily regularised, minimal network whose advantage over simpler models cannot be established from 9 test samples.
2. Scaling and regularization are not optional for MLPs. Both features and target were standardized (`TransformedTargetRegressor`), and the optimal alpha grew by three orders of magnitude (1e-4 to 0.1) when moving from the large to the small data set.
3. Interpretability is lost. Neither the MLP nor the RF gives per-feature effects the way OLS/LASSO coefficients do. The RF at least offers importances, the MLP offers nothing directly.

### Limitations

- The Manufacturing test set is only 9 samples. Differences of a few tenths in RMSE between models are within noise and should not be treated as a ranking. The same applies to the CV preference for tanh over ReLU.
- `MLPRegressor` results depend on `random_state` (weight initialization). A different seed shifts the numbers slightly. The qualitative conclusions above are stable, but a thorough study would average over several seeds as we did in Exercise 7.
- The Polymerization test block is the chronological future, so its error also absorbs any process drift.